#Claims & Risk Analytics

## Purpose

This notebook transforms validated Silver-layer insurance data into business-ready analytical tables for evaluating claim frequency and risk exposure.

## Why is this important?

- **Risk Segmentation:** Identify driver and vehicle characteristics associated with higher claim frequency.
- **Business KPIs:** Calculate reported claims, total exposure, and claim frequency.
- **Decision Support:** Provide consistent metrics for insurance risk analysis.
- **Data Reliability:** Reconcile aggregated Gold metrics with their Silver sources.
- **Analytics Readiness:** Create Delta tables for downstream SQL queries and dashboards.

## Expected Outcome

The notebook produces Gold-layer Delta tables containing aggregated insurance risk metrics, starting with claim frequency by driver age group.

**Note:** Claim frequency is calculated from reported claim counts in the Policies dataset. Claim severity and financial KPIs will be analyzed separately because the Claims dataset does not contain financial records for every reported claim.

In [0]:
# Import PySpark SQL functions
from pyspark.sql import functions as F

# Load validated policy records from the Silver layer
df_policies = spark.table(
    "auto_insurance.silver.policies"
)

# Verify the number of available policy records
print("Total policies:", df_policies.count())

Total policies: 678013


In [0]:
# Define driver age groups for insurance risk analysis
df_policies_segmented = (
    df_policies
    .withColumn(
        "driver_age_group",
        F.when(F.col("driver_age") < 25, "18-24")
         .when(F.col("driver_age") < 35, "25-34")
         .when(F.col("driver_age") < 45, "35-44")
         .when(F.col("driver_age") < 55, "45-54")
         .when(F.col("driver_age") < 65, "55-64")
         .otherwise("65+")
    )
)

# Preview the segmented policy data
display(
    df_policies_segmented.select(
        "policy_id",
        "driver_age",
        "driver_age_group",
        "claim_count",
        "exposure"
    ).limit(10)
)

policy_id,driver_age,driver_age_group,claim_count,exposure
1,55,55-64,1,0.1
3,55,55-64,1,0.77
5,52,45-54,1,0.75
10,46,45-54,1,0.09
11,46,45-54,1,0.84
13,38,35-44,1,0.52
15,38,35-44,1,0.45
17,33,25-34,1,0.27
18,33,25-34,1,0.71
21,41,35-44,1,0.15


## Business Question 1 — Claim Frequency by Driver Age

### Business Question

**Which driver age groups have the highest claim frequency?**

### KPI Definition

**Claim Frequency = Total Reported Claims / Total Exposure**

Claim frequency measures the number of reported claims per policy-year of insurance exposure. Multiplying the result by 100 expresses the metric as claims per 100 policy-years.

### Data Sources

- **Source:** `auto_insurance.silver.policies`
- **Gold Table:** `auto_insurance.gold.risk_by_driver_age`
- **Dimensions:** `driver_age_group`
- **Metrics:** `total_policies`, `total_exposure`, `total_claims`, `claim_frequency`, `claim_frequency_pct`

In [0]:
# Aggregate reported claims and exposure by driver age group
df_risk_by_driver_age = (
    df_policies_segmented
    .groupBy("driver_age_group")
    .agg(
        F.count("*").alias("total_policies"),
        F.sum("exposure").alias("total_exposure"),
        F.sum("claim_count").alias("total_claims")
    )
    .withColumn(
        "claim_frequency",
        F.when(
            F.col("total_exposure") > 0,
            F.col("total_claims") / F.col("total_exposure")
        )
    )
    .withColumn(
        "claim_frequency_pct",
        F.round(F.col("claim_frequency") * 100, 2)
    )
)

# Display claim frequency by driver age group
display(
    df_risk_by_driver_age.orderBy("driver_age_group")
)

driver_age_group,total_policies,total_exposure,total_claims,claim_frequency,claim_frequency_pct
18-24,30198,12489.161084606209,2364,0.189284130774308,18.93
25-34,140947,64598.6144365315,6396,0.09901141155719534,9.9
35-44,170578,87391.45251647146,8145,0.09320133451798204,9.32
45-54,164034,89852.74648235393,9513,0.10587322449701908,10.59
55-64,99094,56176.95111151418,5124,0.09121178523605869,9.12
65+,73162,47990.51983076003,4560,0.09501876654141221,9.5


In [0]:
# Save driver age risk metrics as a managed Delta table
(
    df_risk_by_driver_age.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.risk_by_driver_age"
    )
)

print("Gold table created successfully.")

Gold table created successfully.


In [0]:
# Load the persisted Gold table
df_gold = spark.table(
    "auto_insurance.gold.risk_by_driver_age"
)

# Calculate Gold-level totals
gold_totals = df_gold.agg(
    F.sum("total_policies").alias("policies"),
    F.sum("total_claims").alias("claims"),
    F.sum("total_exposure").alias("exposure")
).first()

# Calculate source totals from Silver
silver_totals = df_policies.agg(
    F.count("*").alias("policies"),
    F.sum("claim_count").alias("claims"),
    F.sum("exposure").alias("exposure")
).first()

# Validate policy and reported claim totals
assert gold_totals["policies"] == silver_totals["policies"]
assert gold_totals["claims"] == silver_totals["claims"]

# Validate exposure using a floating-point tolerance
assert abs(
    gold_totals["exposure"] - silver_totals["exposure"]
) < 0.001

print("Gold reconciliation completed successfully.")

Gold reconciliation completed successfully.


### Key Findings

- **Drivers aged 18–24** have the highest claim frequency at **18.93 claims per 100 policy-years**.
- **Drivers aged 45–54** have the second-highest claim frequency at **10.59 claims per 100 policy-years**.
- **Drivers aged 55–64** have the lowest observed claim frequency at **9.12 claims per 100 policy-years**.
- The claim frequency for drivers aged 18–24 is approximately **2.08 times higher** than for drivers aged 55–64.

### Business Interpretation

The results indicate that younger drivers (18–24) represent the highest observed claim-frequency segment in this dataset.

These findings can support risk segmentation, underwriting analysis, and further investigation of driver characteristics associated with insurance claims.

However, the analysis is descriptive and does not establish a causal relationship between driver age and claim frequency. Other factors, such as vehicle characteristics, geographic location, and bonus-malus history, may influence the observed differences.

### Data Quality & Validation

- **Total Policies:** 678,013
- **Total Reported Claims:** 36,102
- Gold-level policy and claim totals were successfully reconciled with the Silver source data.
- Claim frequency is calculated using reported claim counts from the Policies dataset and is not affected by the incomplete financial claim records identified during claims reconciliation.

### Conclusion

**Drivers aged 18–24 exhibit the highest observed claim frequency, making them a priority segment for further risk analysis.**

-----------------------------

## Business Question 2 — Claim Frequency by Vehicle Age

### Business Question

**Which vehicle age groups have the highest claim frequency?**

### KPI Definition

**Claim Frequency = Total Reported Claims / Total Exposure**

Claim frequency measures the number of reported claims per policy-year of insurance exposure. Multiplying the result by 100 expresses the metric as claims per 100 policy-years.

### Data Sources

- **Source:** `auto_insurance.silver.policies`
- **Gold Table:** `auto_insurance.gold.risk_by_vehicle_age`
- **Dimension:** `vehicle_age_group`
- **Metrics:** `total_policies`, `total_exposure`, `total_claims`, `claim_frequency`, `claim_frequency_pct`

In [0]:
# Define vehicle age groups for risk segmentation
df_vehicle_segmented = (
    df_policies
    .withColumn(
        "vehicle_age_group",
        F.when(F.col("vehicle_age") <= 2, "0-2")
         .when(F.col("vehicle_age") <= 5, "3-5")
         .when(F.col("vehicle_age") <= 10, "6-10")
         .when(F.col("vehicle_age") <= 15, "11-15")
         .otherwise("16+")
    )
)

# Preview vehicle age segmentation
display(
    df_vehicle_segmented.select(
        "policy_id",
        "vehicle_age",
        "vehicle_age_group",
        "claim_count",
        "exposure"
    ).limit(10)
)

policy_id,vehicle_age,vehicle_age_group,claim_count,exposure
1,0,0-2,1,0.1
3,0,0-2,1,0.77
5,2,0-2,1,0.75
10,0,0-2,1,0.09
11,0,0-2,1,0.84
13,2,0-2,1,0.52
15,2,0-2,1,0.45
17,0,0-2,1,0.27
18,0,0-2,1,0.71
21,0,0-2,1,0.15


In [0]:
# Aggregate reported claims and exposure by vehicle age group
df_risk_by_vehicle_age = (
    df_vehicle_segmented
    .groupBy("vehicle_age_group")
    .agg(
        F.count("*").alias("total_policies"),
        F.sum("exposure").alias("total_exposure"),
        F.sum("claim_count").alias("total_claims")
    )
    .withColumn(
        "claim_frequency",
        F.when(
            F.col("total_exposure") > 0,
            F.col("total_claims") / F.col("total_exposure")
        )
    )
    .withColumn(
        "claim_frequency_pct",
        F.round(F.col("claim_frequency") * 100, 2)
    )
)

# Display vehicle age risk indicators
display(
    df_risk_by_vehicle_age.orderBy("vehicle_age_group")
)

vehicle_age_group,total_policies,total_exposure,total_claims,claim_frequency,claim_frequency_pct
0-2,188147,79176.05534743385,10904,0.13771840428462828,13.77
11-15,134063,77990.9754067319,6648,0.08524063156448443,8.52
16+,51719,30283.374571401106,1961,0.06475500262945998,6.48
3-5,132490,72040.12377633926,6743,0.093600616524963,9.36
6-10,171594,99008.91636034776,9846,0.09944558896256378,9.94


In [0]:
# Save vehicle age risk metrics as a managed Delta table
(
    df_risk_by_vehicle_age.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.risk_by_vehicle_age"
    )
)

print("Gold vehicle age risk table created successfully.")

Gold vehicle age risk table created successfully.


In [0]:
# Load the persisted Gold table
df_vehicle_gold = spark.table(
    "auto_insurance.gold.risk_by_vehicle_age"
)

# Calculate Gold-level totals
vehicle_gold_totals = df_vehicle_gold.agg(
    F.sum("total_policies").alias("policies"),
    F.sum("total_claims").alias("claims"),
    F.sum("total_exposure").alias("exposure")
).first()

# Validate against Silver source totals
assert vehicle_gold_totals["policies"] == silver_totals["policies"]
assert vehicle_gold_totals["claims"] == silver_totals["claims"]

# Allow a small tolerance for floating-point aggregation
assert abs(
    vehicle_gold_totals["exposure"] - silver_totals["exposure"]
) < 0.001

print("Vehicle age Gold reconciliation completed successfully.")

Vehicle age Gold reconciliation completed successfully.


### Key Findings

- **Vehicles aged 0–2 years** have the highest claim frequency at **13.77 claims per 100 policy-years**.
- **Vehicles aged 6–10 years** have a claim frequency of **9.94 claims per 100 policy-years**.
- **Vehicles aged 16+ years** have the lowest observed claim frequency at **6.48 claims per 100 policy-years**.
- Vehicles aged 0–2 years have approximately **2.13 times the claim frequency** of vehicles aged 16+ years.
- Claim frequency does not decrease consistently across all vehicle age groups.

### Business Interpretation

The analysis indicates that newer vehicles have the highest observed claim frequency in this dataset.

These findings may support insurance risk segmentation, underwriting analysis, and further investigation of vehicle-related risk factors.

However, this is a descriptive analysis. Differences in claim frequency may also be influenced by driver characteristics, geographic location, vehicle usage, and other variables. The results do not establish a causal relationship between vehicle age and claim frequency.

### Data Quality & Validation

- **Total Policies:** 678,013
- **Total Reported Claims:** 36,102
- Aggregated policy and reported claim counts match the Silver source totals.
- Claim frequency uses reported claim counts from the Policies dataset, independently of the incomplete financial claim records identified during reconciliation.

### Conclusion

**Vehicles aged 0–2 years exhibit the highest observed claim frequency, while vehicles aged 16+ years exhibit the lowest. Vehicle age is therefore a useful dimension for further insurance risk analysis.**

%md
-----------------------------

## Business Question 3 — Claim Frequency by Region

### Business Question

**Which geographic regions have the highest claim frequency?**

### KPI Definition

**Claim Frequency = Total Reported Claims / Total Exposure**

Claim frequency measures the number of reported claims per policy-year of insurance exposure. Multiplying the result by 100 expresses the metric as claims per 100 policy-years.

### Data Sources

- **Source:** `auto_insurance.silver.policies`
- **Gold Table:** `auto_insurance.gold.risk_by_region`
- **Dimension:** `region`
- **Metrics:** `total_policies`, `total_exposure`, `total_claims`, `claim_frequency`, `claim_frequency_pct`

In [0]:
# Aggregate reported claims and exposure by geographic region
df_risk_by_region = (
    df_policies
    .groupBy("region")
    .agg(
        F.count("*").alias("total_policies"),
        F.sum("exposure").alias("total_exposure"),
        F.sum("claim_count").alias("total_claims")
    )
    .withColumn(
        "claim_frequency",
        F.when(
            F.col("total_exposure") > 0,
            F.col("total_claims") / F.col("total_exposure")
        )
    )
    .withColumn(
        "claim_frequency_pct",
        F.round(F.col("claim_frequency") * 100, 2)
    )
)

# Display regions ranked by highest claim frequency
display(
    df_risk_by_region.orderBy(
        F.col("claim_frequency").desc()
    )
)

region,total_policies,total_exposure,total_claims,claim_frequency,claim_frequency_pct
Corse,4516,1765.9353993509844,247,0.13986921610540073,13.99
Champagne-Ardenne,3026,1205.831509091995,160,0.13268852140087287,13.27
Ile-de-France,69791,30208.641954402596,3978,0.13168417190036072,13.17
Picardie,7994,3574.4102208109844,439,0.12281746438728483,12.28
Rhone-Alpes,84752,45347.417285633994,5032,0.11096552573886344,11.1
Alsace,2200,1209.1158230379965,133,0.10999773343948743,11.0
Provence-Alpes-Cotes-D'Azur,79315,35790.32101496556,3907,0.10916359197690086,10.92
Limousin,4567,2395.99454224799,259,0.10809707427672137,10.81
Languedoc-Roussillon,35805,14736.104176928493,1547,0.10498025675076678,10.5
Bourgogne,10492,5025.358609158937,511,0.10168428558882943,10.17


In [0]:
# Persist regional risk indicators as a managed Delta table
(
    df_risk_by_region.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.risk_by_region"
    )
)

print("Gold regional risk table created successfully.")

Gold regional risk table created successfully.


In [0]:
# Load the persisted regional Gold table
df_region_gold = spark.table(
    "auto_insurance.gold.risk_by_region"
)

# Calculate aggregated Gold totals
region_gold_totals = df_region_gold.agg(
    F.sum("total_policies").alias("policies"),
    F.sum("total_claims").alias("claims"),
    F.sum("total_exposure").alias("exposure")
).first()

# Reconcile regional Gold metrics with Silver source totals
assert region_gold_totals["policies"] == silver_totals["policies"]
assert region_gold_totals["claims"] == silver_totals["claims"]

# Validate exposure with a floating-point tolerance
assert abs(
    region_gold_totals["exposure"] - silver_totals["exposure"]
) < 0.001

print("Regional Gold reconciliation completed successfully.")

Regional Gold reconciliation completed successfully.


### Key Findings

- **Corse** has the highest observed claim frequency at **13.99 claims per 100 policy-years**, based on 4,516 policies.
- **Champagne-Ardenne** ranks second at **13.27 claims per 100 policy-years**.
- **Ile-de-France** has a claim frequency of **13.17 claims per 100 policy-years**, with a substantial portfolio of 69,791 policies.
- **Auvergne** has the lowest observed claim frequency at **8.14 claims per 100 policy-years**.
- **Centre** has the largest policy portfolio (160,601 policies) and the highest total reported claims (9,204), but a relatively low claim frequency of **8.96 claims per 100 policy-years**.

### Business Interpretation

Regional claim frequency varies across the insurance portfolio.

Ile-de-France is a particularly relevant region for further risk analysis because it combines high observed claim frequency with substantial insurance exposure.

Although Corse has the highest claim frequency, its smaller portfolio size means that the estimate may be more sensitive to variation.

These results can support geographic risk segmentation and underwriting analysis. However, regional differences may also reflect variations in driver profiles, vehicle characteristics, and other factors.

### Data Quality & Validation

- **Total Policies:** 678,013
- **Total Reported Claims:** 36,102
- **Regions Analyzed:** 21
- Aggregated policy and reported claim counts match the Silver source totals.
- Claim frequency is calculated from reported claim counts in the Policies dataset, independently of financial claim record availability.

### Conclusion

**Corse has the highest observed claim frequency, while Ile-de-France stands out as a high-frequency region with substantial portfolio exposure. Regional segmentation provides valuable insights for further insurance risk analysis.**

----------------------

In [0]:
# Load validated financial claim records from the Silver layer
df_claims = spark.table(
    "auto_insurance.silver.claims"
)

# Verify the number of available claim records
print("Total valid claim records:", df_claims.count())

Total valid claim records: 26444


## Business Question 4 — Average Claim Severity by Driver Age

### Business Question

**Which driver age groups have the highest average claim severity?**

### KPI Definition

**Average Claim Severity = Total Recorded Claim Amount / Number of Recorded Claims**

Average claim severity measures the average financial amount associated with an available claim record.

### Data Sources

- **Sources:** `auto_insurance.silver.claims`, `auto_insurance.silver.policies`
- **Gold Table:** `auto_insurance.gold.severity_by_driver_age`
- **Dimension:** `driver_age_group`
- **Metrics:** `recorded_claims`, `total_claim_amount`, `average_claim_severity`

In [0]:
# Join financial claim records with policy driver age segments
df_claims_with_driver_age = (
    df_claims
    .join(
        df_policies_segmented.select(
            "policy_id",
            "driver_age_group"
        ),
        on="policy_id",
        how="inner"
    )
)

# Preview enriched financial claim records
display(
    df_claims_with_driver_age.select(
        "policy_id",
        "driver_age_group",
        "claim_amount"
    ).limit(10)
)

policy_id,driver_age_group,claim_amount
139,55-64,303.0
190,45-54,1981.84
414,35-44,1456.55
424,45-54,9844.36
463,45-54,3986.67
606,45-54,1840.14
622,25-34,1397.97
811,35-44,971.98
830,18-24,1442.75
975,55-64,637.41


In [0]:
# Aggregate recorded claim amounts by driver age group
df_severity_by_driver_age = (
    df_claims_with_driver_age
    .groupBy("driver_age_group")
    .agg(
        F.count("*").alias("recorded_claims"),
        F.sum("claim_amount").alias("total_claim_amount"),
        F.avg("claim_amount").alias("average_claim_severity")
    )
    .withColumn(
        "average_claim_severity",
        F.round(F.col("average_claim_severity"), 2)
    )
    .withColumn(
        "total_claim_amount",
        F.round(F.col("total_claim_amount"), 2)
    )
)

# Display driver age groups ranked by average claim severity
display(
    df_severity_by_driver_age.orderBy(
        F.col("average_claim_severity").desc()
    )
)

driver_age_group,recorded_claims,total_claim_amount,average_claim_severity
18-24,2011,1.174134421E7,5838.56
65+,2864,6658804.38,2325.0
25-34,4973,9957419.91,2002.3
35-44,6192,1.228448134E7,1983.93
55-64,3602,6697343.57,1859.34
45-54,6802,1.256982309E7,1847.96


In [0]:
# Save claim severity metrics as a managed Delta table
(
    df_severity_by_driver_age.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.severity_by_driver_age"
    )
)

print("Gold claim severity table created successfully.")

Gold claim severity table created successfully.


In [0]:
# Load the persisted Gold severity table
df_severity_gold = spark.table(
    "auto_insurance.gold.severity_by_driver_age"
)

# Calculate Gold-level financial claim totals
gold_severity_totals = df_severity_gold.agg(
    F.sum("recorded_claims").alias("claims"),
    F.sum("total_claim_amount").alias("amount")
).first()

# Calculate Silver-level financial claim totals
silver_severity_totals = df_claims.agg(
    F.count("*").alias("claims"),
    F.sum("claim_amount").alias("amount")
).first()

# Validate the number of recorded financial claims
assert gold_severity_totals["claims"] == silver_severity_totals["claims"]

# Validate financial totals with a tolerance for rounding
assert abs(
    gold_severity_totals["amount"] - silver_severity_totals["amount"]
) < 0.1

print("Claim severity Gold reconciliation completed successfully.")

Claim severity Gold reconciliation completed successfully.


### Key Findings

- **Drivers aged 18–24** have the highest average claim severity at **5,838.56**.
- **Drivers aged 65+** have the second-highest average severity at **2,325.00**.
- **Drivers aged 45–54** have the lowest average severity at **1,847.96**.
- Average claim severity for drivers aged 18–24 is approximately **3.16 times higher** than for drivers aged 45–54.
- The 18–24 segment also has the highest reported claim frequency at **18.93 claims per 100 policy-years**.

### Business Interpretation

The results indicate that drivers aged 18–24 have both the highest observed claim frequency and the highest average severity among available financial claim records.

This makes younger drivers a relevant segment for further insurance risk analysis.

However, claim severity may be influenced by large individual claims, differences in claim composition, and incomplete financial claim coverage. These findings should not be interpreted as causal relationships.

### Data Quality & Validation

- **Total Recorded Financial Claims:** 26,444
- **Total Recorded Claim Amount:** 59,909,216.50
- All 26,444 valid financial claim records are represented in the age-group aggregation.
- The Policies dataset reports 36,102 claims, but only 26,444 linked financial claim records are available.
- Average claim severity therefore represents the available recorded claims, not necessarily the full population of reported claims.
- The monetary currency is not explicitly established by the source data.

### Conclusion

**Drivers aged 18–24 exhibit the highest average claim severity and the highest reported claim frequency. This segment warrants further investigation, while accounting for incomplete financial coverage and potential high-value claim outliers.**

-------------------------------------

## Business Question 5 — Claim Frequency by Vehicle Characteristics

### Business Question

**Which vehicle characteristics are associated with higher claim frequency?**

### KPI Definition

**Claim Frequency = Total Reported Claims / Total Exposure**

Claim frequency measures the number of reported claims per policy-year of insurance exposure. Multiplying the result by 100 expresses the metric as claims per 100 policy-years.

### Data Sources

- **Source:** `auto_insurance.silver.policies`
- **Gold Tables:**
  - `auto_insurance.gold.risk_by_vehicle_power`
  - `auto_insurance.gold.risk_by_vehicle_fuel`
- **Dimensions:** `vehicle_power`, `vehicle_fuel`
- **Metrics:** `total_policies`, `total_exposure`, `total_claims`, `claim_frequency`, `claim_frequency_pct`



In [0]:
# Aggregate reported claims and exposure by vehicle power category
df_risk_by_vehicle_power = (
    df_policies
    .groupBy("vehicle_power")
    .agg(
        F.count("*").alias("total_policies"),
        F.sum("exposure").alias("total_exposure"),
        F.sum("claim_count").alias("total_claims")
    )
    .withColumn(
        "claim_frequency",
        F.when(
            F.col("total_exposure") > 0,
            F.col("total_claims") / F.col("total_exposure")
        )
    )
    .withColumn(
        "claim_frequency_pct",
        F.round(F.col("claim_frequency") * 100, 2)
    )
)

# Display vehicle power categories ranked by claim frequency
display(
    df_risk_by_vehicle_power.orderBy(
        F.col("claim_frequency").desc()
    )
)

vehicle_power,total_policies,total_exposure,total_claims,claim_frequency,claim_frequency_pct
10,31354,15375.010824124962,1789,0.11635764166051034,11.64
9,30085,15348.21365592322,1754,0.11428040026815056,11.43
5,124821,68173.34882087677,7278,0.1067572610980679,10.68
11,18352,8497.778451957682,897,0.10555699999372813,10.56
6,148976,82524.21017201089,8381,0.10155807589713255,10.16
14,2350,1182.3503907430056,119,0.10064698327305388,10.06
15,2926,1260.5291728379946,124,0.09837138455179308,9.84
7,145401,77950.45670019973,7627,0.09784419903187633,9.78
4,115349,60073.63730356177,5699,0.09486690428285596,9.49
12,8214,3792.0383426809685,359,0.09467203850744485,9.47


In [0]:
# Aggregate reported claims and exposure by vehicle fuel type
df_risk_by_vehicle_fuel = (
    df_policies
    .groupBy("vehicle_fuel")
    .agg(
        F.count("*").alias("total_policies"),
        F.sum("exposure").alias("total_exposure"),
        F.sum("claim_count").alias("total_claims")
    )
    .withColumn(
        "claim_frequency",
        F.when(
            F.col("total_exposure") > 0,
            F.col("total_claims") / F.col("total_exposure")
        )
    )
    .withColumn(
        "claim_frequency_pct",
        F.round(F.col("claim_frequency") * 100, 2)
    )
)

# Display fuel types ranked by claim frequency
display(
    df_risk_by_vehicle_fuel.orderBy(
        F.col("claim_frequency").desc()
    )
)

vehicle_fuel,total_policies,total_exposure,total_claims,claim_frequency,claim_frequency_pct
Regular,345877,187838.5523358215,19454,0.1035676636030486,10.36
Diesel,332136,170660.89312635682,16648,0.09755017505781989,9.76


In [0]:
# Save vehicle power risk metrics
(
    df_risk_by_vehicle_power.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.risk_by_vehicle_power"
    )
)

# Save vehicle fuel risk metrics
(
    df_risk_by_vehicle_fuel.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.risk_by_vehicle_fuel"
    )
)

print("Vehicle characteristics Gold tables created successfully.")

Vehicle characteristics Gold tables created successfully.


In [0]:
# Validate vehicle power and fuel Gold tables against Silver totals
for table_name in [
    "auto_insurance.gold.risk_by_vehicle_power",
    "auto_insurance.gold.risk_by_vehicle_fuel"
]:
    df_gold = spark.table(table_name)

    gold_totals = df_gold.agg(
        F.sum("total_policies").alias("policies"),
        F.sum("total_claims").alias("claims"),
        F.sum("total_exposure").alias("exposure")
    ).first()

    # Verify that all policies and reported claims are preserved
    assert gold_totals["policies"] == silver_totals["policies"]
    assert gold_totals["claims"] == silver_totals["claims"]

    # Verify exposure totals with a floating-point tolerance
    assert abs(
        gold_totals["exposure"] - silver_totals["exposure"]
    ) < 0.001

    print(f"Validation passed: {table_name}")

Validation passed: auto_insurance.gold.risk_by_vehicle_power
Validation passed: auto_insurance.gold.risk_by_vehicle_fuel


### Key Findings

**Vehicle Power**

- Vehicle power category **10** has the highest observed claim frequency at **11.64 claims per 100 policy-years**.
- Vehicle power category **9** ranks second at **11.43 claims per 100 policy-years**.
- Vehicle power category **8** has the lowest observed claim frequency at **8.47 claims per 100 policy-years**.
- Claim frequency does not increase consistently with vehicle power category.
- Some higher power categories have relatively low exposure, requiring cautious interpretation.

**Vehicle Fuel**

- **Regular:** 10.36 claims per 100 policy-years.
- **Diesel:** 9.76 claims per 100 policy-years.
- Regular-fuel vehicles have approximately **6.2% higher observed claim frequency** than diesel vehicles.

### Business Interpretation

Vehicle power and fuel type are associated with differences in observed claim frequency.

Vehicle power categories 10 and 9 exhibit the highest frequencies, but the relationship between vehicle power and claim frequency is not monotonic.

Regular-fuel vehicles show a slightly higher claim frequency than diesel vehicles.

These results may support insurance risk segmentation and further investigation of vehicle characteristics. However, differences may be influenced by driver demographics, vehicle age, geographic location, and other factors.

The analysis is descriptive and does not establish causality.

### Data Quality & Validation

- **Total Policies:** 678,013
- **Total Reported Claims:** 36,102
- **Vehicle Power Categories:** 12
- **Vehicle Fuel Categories:** 2
- Both Gold aggregations preserve the total policy and reported claim counts from the Silver layer.
- Claim frequency is calculated using reported claim counts from the Policies dataset, independently of financial claim record availability.

### Conclusion

**Vehicle power category 10 has the highest observed claim frequency, while Regular-fuel vehicles show a slightly higher frequency than Diesel vehicles. Both characteristics provide useful dimensions for insurance risk segmentation, although further multivariable analysis is needed to understand the underlying differences.**

---------------------------------------

## Business Question 6 — Observed Claims Cost per Exposure by Driver Age

### Business Question

**Which driver age groups have the highest observed claims cost per exposure?**

### KPI Definition

**Observed Claims Cost per Exposure = Total Recorded Claim Amount / Total Policy Exposure**

This KPI measures the financial amount of available recorded claims per policy-year of insurance exposure.

It does not represent ultimate pure premium because financial claim records are not available for every reported claim.

### Data Sources

- **Sources:** `auto_insurance.silver.policies`, `auto_insurance.silver.claims`
- **Gold Table:** `auto_insurance.gold.claims_cost_by_driver_age`
- **Dimension:** `driver_age_group`
- **Metrics:** `total_policies`, `total_exposure`, `total_claims`, `recorded_claims`, `total_claim_amount`, `observed_claims_cost_per_exposure`



In [0]:
# Aggregate available financial claim amounts by driver age group
df_claim_cost_by_age = (
    df_claims_with_driver_age
    .groupBy("driver_age_group")
    .agg(
        F.count("*").alias("recorded_claims"),
        F.sum("claim_amount").alias("total_claim_amount")
    )
)

# Preview financial claim totals
display(
    df_claim_cost_by_age.orderBy("driver_age_group")
)

driver_age_group,recorded_claims,total_claim_amount
18-24,2011,1.174134421000001E7
25-34,4973,9957419.910000013
35-44,6192,1.2284481340000063E7
45-54,6802,1.2569823090000072E7
55-64,3602,6697343.570000011
65+,2864,6658804.380000025


In [0]:
# Join policy exposure metrics with available financial claim amounts
df_cost_per_exposure = (
    df_risk_by_driver_age
    .select(
        "driver_age_group",
        "total_policies",
        "total_exposure",
        "total_claims"
    )
    .join(
        df_claim_cost_by_age,
        on="driver_age_group",
        how="left"
    )
    .fillna({
        "recorded_claims": 0,
        "total_claim_amount": 0.0
    })
    .withColumn(
        "observed_claims_cost_per_exposure",
        F.when(
            F.col("total_exposure") > 0,
            F.col("total_claim_amount") / F.col("total_exposure")
        )
    )
    .withColumn(
        "observed_claims_cost_per_exposure",
        F.round(F.col("observed_claims_cost_per_exposure"), 2)
    )
)

# Display age groups ranked by observed claims cost per exposure
display(
    df_cost_per_exposure.orderBy(
        F.col("observed_claims_cost_per_exposure").desc()
    )
)

driver_age_group,total_policies,total_exposure,total_claims,recorded_claims,total_claim_amount,observed_claims_cost_per_exposure
18-24,30198,12489.161084606209,2364,2011,1.174134421000001E7,940.12
25-34,140947,64598.6144365315,6396,4973,9957419.910000013,154.14
35-44,170578,87391.45251647146,8145,6192,1.2284481340000063E7,140.57
45-54,164034,89852.74648235393,9513,6802,1.2569823090000072E7,139.89
65+,73162,47990.51983076003,4560,2864,6658804.380000025,138.75
55-64,99094,56176.95111151418,5124,3602,6697343.570000011,119.22


In [0]:
# Save observed claims cost metrics as a managed Delta table
(
    df_cost_per_exposure.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "auto_insurance.gold.claims_cost_by_driver_age"
    )
)

print("Gold observed claims cost table created successfully.")

Gold observed claims cost table created successfully.


In [0]:
# Load the persisted Gold table
df_cost_gold = spark.table(
    "auto_insurance.gold.claims_cost_by_driver_age"
)

# Aggregate Gold metrics
gold_cost_totals = df_cost_gold.agg(
    F.sum("total_policies").alias("policies"),
    F.sum("total_claims").alias("reported_claims"),
    F.sum("recorded_claims").alias("recorded_claims"),
    F.sum("total_exposure").alias("exposure"),
    F.sum("total_claim_amount").alias("amount")
).first()

# Aggregate Silver policy metrics
silver_policy_totals = df_policies.agg(
    F.count("*").alias("policies"),
    F.sum("claim_count").alias("reported_claims"),
    F.sum("exposure").alias("exposure")
).first()

# Aggregate Silver financial claim metrics
silver_claim_totals = df_claims.agg(
    F.count("*").alias("recorded_claims"),
    F.sum("claim_amount").alias("amount")
).first()

# Validate policy and claim record counts
assert gold_cost_totals["policies"] == silver_policy_totals["policies"]
assert gold_cost_totals["reported_claims"] == silver_policy_totals["reported_claims"]
assert gold_cost_totals["recorded_claims"] == silver_claim_totals["recorded_claims"]

# Validate exposure and financial amounts
assert abs(
    gold_cost_totals["exposure"] - silver_policy_totals["exposure"]
) < 0.001

assert abs(
    gold_cost_totals["amount"] - silver_claim_totals["amount"]
) < 0.01

print("Observed claims cost Gold reconciliation completed successfully.")

Observed claims cost Gold reconciliation completed successfully.


### Key Findings

- **Drivers aged 18–24** have the highest observed claims cost per exposure at **940.12**.
- **Drivers aged 25–34** rank second at **154.14**.
- **Drivers aged 55–64** have the lowest observed cost per exposure at **119.22**.
- The observed cost per exposure for drivers aged 18–24 is approximately **7.89 times higher** than for drivers aged 55–64.
- The 18–24 segment also has the highest reported claim frequency and average recorded claim severity.

### Business Interpretation

Drivers aged 18–24 exhibit the highest observed financial claims cost relative to insurance exposure.

The results suggest that this segment warrants further investigation, particularly regarding large individual claims and differences in financial claim coverage.

Observed claims cost per exposure should not be interpreted as a complete insurance pricing metric or ultimate pure premium.

Because reported claim counts and available financial claim records do not fully reconcile, the financial cost metric may underestimate total claims costs, and comparisons across age groups may be affected by differences in financial record availability.

### Data Quality & Validation

- **Total Policies:** 678,013
- **Total Reported Claims:** 36,102
- **Total Recorded Financial Claims:** 26,444
- **Total Recorded Claim Amount:** 59,909,216.50
- Gold policy counts, reported claims, and recorded financial claim counts reconcile with their Silver sources.
- The financial metric includes only available claims linked to valid policies.
- The monetary currency is not explicitly established by the source data.

### Conclusion

**Drivers aged 18–24 have the highest observed claims cost per exposure, substantially exceeding all other driver age groups. This finding highlights a priority segment for further risk analysis, subject to the limitations of incomplete financial claim coverage and potential high-value outliers.**